## DLA CODE OF CONDUCT V2.0

This Code of Conduct defines the principles governing ethical, transparent, and responsible use of Large Language Models (LLMs), online resources, and peer collaboration in the Deep Learning Applications laboratories. This version of the Code of Conduct was refined via a brainstorming session with **ChatGPT Version 5.2** and subsequently adapted to reflect the specific requirements and values of the DLA laboratories. In that spirit, this Code itself models the transparency it expects from you.

***Our goal is not to restrict innovation, but to ensure integrity, accountability, and genuine learning.***

### 1. Transparency in the Use of LLMs and AI Tools

The use of LLMs and AI-assisted tools is permitted — *but it must be transparent*.

* **Explicit Disclosure:** Clearly state if and how LLMs (e.g., ChatGPT, Copilot, Claude, etc.) were used. This includes code generation, debugging, data analysis, experiment design, report writing, or conceptual clarification.
* **Description of Contribution:** Briefly describe what the tool contributed and how you modified, verified, or extended its output.
* **Acknowledgment of Limitations:** Recognize that LLM outputs may contain errors, biases, or non-optimal solutions. You are responsible for verifying correctness, appropriateness, and academic integrity.

***Using AI does not reduce your responsibility for the final result.***

### 2. Proper Attribution and Documentation

Deep learning builds on existing work — responsibly.

* **Attribution:** Properly cite all external resources, including: Code snippets, Tutorials, Documentation, Datasets, Pretrained models, Research papers, and AI-generated content.
* **Reproducibility:** Clearly document tools, libraries, model versions, hyperparameters, and experimental setups so that your work can be reproduced.
* **Clarity of Modifications:** If you adapt external code, explicitly indicate what you changed and why.

***Transparency is a sign of scientific maturity — not weakness.***

### 3. Collaboration and Individual Responsibility

Discussion is encouraged. Copying is not.

* **Collaborative Learning:** You are encouraged to discuss concepts, debugging strategies, and approaches with classmates.
* **Individual Submission:** Your submitted solution must reflect your own understanding and implementation.
* **No Direct Sharing of Solutions:** Do not share complete solutions, trained models, or reports. Do not submit another person's work — or AI-generated work — as your own without meaningful engagement and proper disclosure.

***If you cannot explain your submission, it is not your submission.***

### 4. Accountability and Academic Integrity

You are responsible for everything you submit. Failure to comply with these guidelines may result in review by the course examination commission and can lead to disciplinary measures in accordance with university regulations.

***Integrity is part of your training as a machine learning practitioner.***

### 5. The Spirit of This Code of Conduct

This course prepares you to work in a field where:

* Reproducibility matters
* Ethical considerations matter
* Transparency matters
* Responsible AI use matters

***The purpose of this Code of Conduct is not surveillance — it is professional formation.***

### TL;DR

Use AI; Don’t let AI use you; Be transparent; Cite everything; Do your own thinking.

***If you can’t explain it, you probably shouldn’t submit it.***

---
---

## Introduction

In this second laboratory we will gain some experience working with Transformer models for a variety tasks using (mostly) the Hugging Face Ecosystem. 


---
### Exercise 1: Sentiment Analysis (warm up)

In this first exercise we will start from a pre-trained BERT transformer and build up a model able to perform text sentiment analysis. Transformers are complex beasts, so we will build up our pipeline in several explorative and incremental steps.

#### Exercise 1.1: Loading the Dataset Splits
There are a many sentiment analysis datasets, but we will use one of the smallest ones available: the [Cornell Rotten Tomatoes movie review dataset](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes), which consists of 5,331 positive and 5,331 negative processed sentences from the Rotten Tomatoes movie reviews.

**Your first task**: Load the dataset and figure out what splits are available and how to get them. Spend some time exploring the dataset to see how it is organized. Note that we will be using the [HuggingFace Datasets](https://huggingface.co/docs/datasets/en/index) library for downloading, accessing, splitting, and batching data for training and evaluation.

In [1]:
# Dataset imports.
from datasets import load_dataset, get_dataset_split_names

# Your code here.
splits = get_dataset_split_names("cornell-movie-review-data/rotten_tomatoes")
print(splits)

['train', 'validation', 'test']


In [2]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes", split=splits)
print(ds)

[Dataset({
    features: ['text', 'label'],
    num_rows: 8530
}), Dataset({
    features: ['text', 'label'],
    num_rows: 1066
}), Dataset({
    features: ['text', 'label'],
    num_rows: 1066
})]


In [3]:
ds= {split: load_dataset("cornell-movie-review-data/rotten_tomatoes", split=split) for split in splits }
print(ds)

{'train': Dataset({
    features: ['text', 'label'],
    num_rows: 8530
}), 'validation': Dataset({
    features: ['text', 'label'],
    num_rows: 1066
}), 'test': Dataset({
    features: ['text', 'label'],
    num_rows: 1066
})}


After downloading the datset I print some random reviews with their relative labels

In [3]:
import numpy as np

np.random.seed(1234)
for row in np.random.permutation(len(ds['train']))[:10]:
    print(f'{ds['train'][row]['label']}: {ds['train'][row]['text']}')

1: an absurdist spider web .
1: the delicious trimmingsarrive early and stay late , filling nearly every minutewith a lighthearted glow , some impudent snickers , and a glorious dose of humankind's liberating ability to triumph over a scrooge or two .
0: evelyn may be based on a true and historically significant story , but the filmmakers have made every effort to disguise it as an unimaginative screenwriter's invention .
1: this is a happy throwback to the time when cartoons were cinema's most idiosyncratic form instead of one of its most predictable .
1: even though it is infused with the sensibility of a video director , it doesn't make for completely empty entertainment
0: sustains its dreamlike glide through a succession of cheesy coincidences and voluptuous cheap effects , not the least of which is rebecca romijn-stamos .
0: a backhanded ode to female camaraderie penned by a man who has little clue about either the nature of women or of friendship .
0: another useless recycling


---
### Exercise 1.2: A Pre-trained BERT and Tokenizer

The model we will use is a *very* small BERT transformer called [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) this model was trained (using self-supervised learning) on the same corpus as BERT but using the full BERT base model as a *teacher*.

**Your next task**: Load the DistilBERT model and corresponding tokenizer. Use the tokenizer on a few samples from the dataset and pass the tokens through the model to see what outputs are provided. I suggest you use the [`AutoModel`](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html) class (and the `from_pretrained()` method) to load the model and `AutoTokenizer` to load the tokenizer).

In [4]:
# AutoClass imports.
from transformers import AutoTokenizer, AutoModel

# Your code here.
model = AutoModel.from_pretrained('distilbert-base-uncased')
print(model)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

Looking at the output we can see tha the vocaboulary has dimension 30522 and that every token has dimension 768. The maximum input text length is 512.

In [5]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
print(tokenizer)

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


I pass a review to the tokeniker to encode and then decode it to see how it works 

In [6]:
row = 2
encoded_text = tokenizer.encode(ds['train'][row]['text'])
print(ds['train'][row]['text'])
print(encoded_text)
encoded_text = tokenizer.encode(ds['train'][row]['text'], return_tensors='pt')
print(encoded_text)
decoded_text = tokenizer.decode(encoded_text)
print(decoded_text)


effective but too-tepid biopic
[101, 4621, 2021, 2205, 1011, 8915, 23267, 16012, 24330, 102]
tensor([[  101,  4621,  2021,  2205,  1011,  8915, 23267, 16012, 24330,   102]])
['[CLS] effective but too - tepid biopic [SEP]']


In [7]:
output = model(encoded_text)
print(output)

BaseModelOutput(last_hidden_state=tensor([[[-0.2706, -0.1265, -0.0500,  ..., -0.3721,  0.2477,  0.3306],
         [ 0.0502,  0.0702, -0.0243,  ..., -0.5188,  0.5020,  0.0597],
         [-0.2193, -0.2208,  0.3721,  ..., -0.3424, -0.3176,  0.8824],
         ...,
         [ 0.3400,  0.1952, -0.1578,  ..., -0.8321,  0.2700, -0.0424],
         [ 0.2094, -0.0349, -0.1674,  ..., -0.4733, -0.1024, -0.4121],
         [ 0.8669,  0.2085, -0.3474,  ...,  0.0160, -0.4736, -0.2089]]],
       grad_fn=<NativeLayerNormBackward0>), hidden_states=None, attentions=None)


In [8]:
print(output.last_hidden_state.shape)

torch.Size([1, 10, 768])


Last hidden state is a tensor that contains an embedding of dimension 768 for each token of the embedded review

In [9]:
batch = tokenizer(ds['train'][:2]['text'], return_tensors='pt', padding=True)
print(batch)

{'input_ids': tensor([[  101,  1996,  2600,  2003, 16036,  2000,  2022,  1996,  7398,  2301,
          1005,  1055,  2047,  1000, 16608,  1000,  1998,  2008,  2002,  1005,
          1055,  2183,  2000,  2191,  1037, 17624,  2130,  3618,  2084,  7779,
         29058,  8625, 13327,  1010,  3744,  1011, 18856, 19513,  3158,  5477,
          4168,  2030,  7112, 16562,  2140,  1012,   102,     0,     0,     0,
             0,     0],
        [  101,  1996,  9882,  2135,  9603, 13633,  1997,  1000,  1996,  2935,
          1997,  1996,  7635,  1000, 11544,  2003,  2061,  4121,  2008,  1037,
          5930,  1997,  2616,  3685, 23613,  6235,  2522,  1011,  3213,  1013,
          2472,  2848,  4027,  1005,  1055,  4423,  4432,  1997,  1046,  1012,
          1054,  1012,  1054,  1012, 23602,  1005,  1055,  2690,  1011,  3011,
          1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

The tokenizer returns an attention mask when giving in input more reviews and setting padding=True

In [10]:
output = model(**batch)
print(output)

BaseModelOutput(last_hidden_state=tensor([[[-0.0332, -0.0168,  0.0194,  ...,  0.0476,  0.5834,  0.3036],
         [-0.0235, -0.0555, -0.3638,  ...,  0.1877,  0.5781, -0.1577],
         [-0.0516, -0.1014, -0.1511,  ...,  0.1503,  0.2649, -0.1575],
         ...,
         [ 0.3688, -0.1147,  0.8428,  ..., -0.0708, -0.0178, -0.2516],
         [ 0.0654, -0.0206,  0.1889,  ...,  0.1159,  0.2323, -0.2404],
         [ 0.0373, -0.0104,  0.1203,  ...,  0.1049,  0.2852, -0.3035]],

        [[-0.2062, -0.0490, -0.4036,  ..., -0.1186,  0.6141,  0.3919],
         [-0.4361, -0.1647, -0.3533,  ...,  0.1086,  0.9478, -0.0272],
         [-0.1164,  0.1690,  0.2698,  ..., -0.1971,  0.4372,  0.2527],
         ...,
         [-0.2341,  0.4810, -0.2634,  ..., -0.3397,  0.2567,  0.1274],
         [ 0.7139,  0.0574, -0.3260,  ...,  0.2041, -0.3800, -0.3343],
         [ 0.5649,  0.2806, -0.0295,  ...,  0.1297, -0.3160, -0.1874]]],
       grad_fn=<NativeLayerNormBackward0>), hidden_states=None, attentions=None)


In [11]:
output.last_hidden_state[0][0]

tensor([-3.3174e-02, -1.6809e-02,  1.9412e-02, -2.5718e-02, -1.3797e-01,
        -3.9617e-01,  3.8300e-01,  5.1176e-01,  2.3082e-02, -5.5535e-02,
        -6.3165e-02, -1.3682e-01, -5.1798e-02,  4.9829e-01,  2.3183e-01,
         2.3796e-01, -3.1169e-01,  2.4725e-01,  2.2794e-01,  4.6447e-02,
        -1.5396e-01, -1.5129e-01,  1.7389e-01, -7.2357e-02,  5.8715e-02,
        -1.8521e-01, -3.6629e-02, -6.7262e-02,  8.1639e-02,  2.5985e-01,
         2.3714e-02,  7.8961e-02, -5.1688e-01, -2.9591e-01,  4.5397e-02,
        -1.1534e-01,  9.6400e-02, -9.3000e-02,  1.0850e-01,  2.8973e-01,
         2.3017e-01,  2.2850e-01,  9.5719e-02,  7.5702e-02, -1.8779e-01,
        -2.2961e-01, -2.5935e+00,  1.4054e-01, -9.0067e-02, -2.9513e-01,
         4.7138e-01,  1.5484e-01, -1.9447e-01,  2.0365e-01,  4.3326e-01,
         1.7800e-01, -3.4895e-01,  3.0172e-01, -5.0001e-02, -3.0881e-02,
         6.4041e-02,  1.6840e-01, -1.4781e-01, -1.9468e-01,  4.4127e-02,
         1.0639e-01, -2.3429e-01,  1.4086e-01, -3.1

In [12]:
no_masking = model(input_ids = batch['input_ids'],)
print(output.last_hidden_state[0][0] - no_masking.last_hidden_state[0][0])

tensor([-3.9576e-03, -5.2613e-02, -2.5270e-02, -1.5604e-02, -3.5386e-02,
        -5.6213e-02,  1.1621e-01,  1.0242e-01,  4.4932e-02,  6.3993e-02,
         3.2607e-02, -6.2503e-02, -5.6915e-02,  8.4685e-02,  4.5280e-02,
         2.6263e-02, -8.7788e-02,  6.8171e-02,  4.5885e-02,  1.0324e-01,
        -3.6101e-03, -2.8034e-02,  7.9817e-04,  2.8474e-02,  2.8748e-04,
        -2.2365e-02,  4.4958e-02, -4.3039e-02,  5.3441e-02,  3.9553e-02,
        -3.3282e-02,  3.2831e-02, -1.4795e-01, -8.8305e-02,  4.3885e-02,
        -5.3107e-02, -3.9129e-02, -9.2470e-02, -2.6895e-02,  8.8814e-02,
        -1.0537e-02,  3.2799e-02, -3.5409e-03,  3.3961e-02, -4.7411e-02,
        -1.0424e-01, -3.7566e-01,  4.5566e-03,  1.0376e-02, -3.5926e-02,
         1.1832e-01,  4.3210e-02, -9.3445e-02,  3.2598e-02,  9.7610e-02,
         9.4663e-02, -9.4913e-02, -8.6609e-02, -3.1334e-02, -3.9544e-02,
        -1.2951e-02,  6.5849e-02, -1.6560e-02, -7.5623e-02,  2.8335e-02,
        -2.6976e-02, -4.9236e-02,  6.2781e-02, -7.3

If we do not pass the attention mask to the model we obtains different results because the model computes the attention even on padding tokens


---
### Exercise 1.3: A Stable Baseline

In this exercise I want you to:
1. Use DistilBERT as a *feature extractor* to extract representations of the text strings from the dataset splits;
2. Train a classifier (your choice, by an SVM from Scikit-learn is an easy choice).
3. Evaluate performance on the validation and test splits.

These results are our *stable baseline* -- the **starting** point on which we will (hopefully) improve in the next exercise.

**Hint**: There are a number of ways to implement the feature extractor, but probably the best is to use a [feature extraction `pipeline`](https://huggingface.co/tasks/feature-extraction). You will need to interpret the output of the pipeline and extract only the `[CLS]` token from the *last* transformer layer. *How can you figure out which output that is?*

As a baseline I utilize a linear SVC fitted on the features extracted by DistilBert

In [87]:
from transformers import pipeline
import torch
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# Your code here.

extractor = pipeline('feature-extraction', model=model, tokenizer=tokenizer)
train_feats = extractor(list(ds['train']['text']), return_tensors='pt')
val_feats = extractor(list(ds['validation']['text']), return_tensors='pt')
test_feats = extractor(list(ds['test']['text']), return_tensors='pt')

I extract only the embeddings of CLS tokens

In [141]:
train_feats = torch.vstack([feat[0][0] for feat in train_feats])
val_feats = torch.vstack([feat[0][0] for feat in val_feats])
test_feats = torch.vstack([feat[0][0] for feat in test_feats])
train_feats_np = train_feats.numpy()
val_feats_np = val_feats.numpy()
test_feats_np = test_feats.numpy()

In [142]:
train_label_np = np.array(ds['train']["label"])
val_label_np = np.array(ds['validation']["label"])
test_label_np = np.array(ds['test']["label"])

I use the validation set to select the best value of C that in my case is 0.1

In [146]:
Cs = [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100, 1000]

best_C = None
best_score = -1

for c in Cs:
    svc = LinearSVC(C = c, random_state = 1234)

    svc.fit(train_feats_np, train_label_np)

    label_pred = svc.predict(val_feats_np)

    score = classification_report(val_label_np, label_pred, output_dict= True)

    print(f"C={c}, accuracy={score['accuracy']:.4f}")

    if score['accuracy'] > best_score:
        best_score = score['accuracy']
        best_C = c

print("Best C:", best_C)
print("Best validation accuracy:", best_score)



C=0.0001, accuracy=0.7758
C=0.001, accuracy=0.8077
C=0.01, accuracy=0.8208
C=0.1, accuracy=0.8274
C=1, accuracy=0.8218
C=10, accuracy=0.8180
C=100, accuracy=0.8180
C=1000, accuracy=0.8189
Best C: 0.1
Best validation accuracy: 0.8273921200750469


In [149]:
svc = svc = LinearSVC(C = best_C, random_state = 1234)

svc.fit(train_feats_np, train_label_np)

label_pred = svc.predict(test_feats_np)

score = classification_report(test_label_np, label_pred)

print(score)



              precision    recall  f1-score   support

           0       0.79      0.82      0.80       533
           1       0.81      0.78      0.79       533

    accuracy                           0.80      1066
   macro avg       0.80      0.80      0.80      1066
weighted avg       0.80      0.80      0.80      1066



I obtain an accuracy of 0.8 on the test set setting c = 0.1

---
---
## Exercise 2: Fine-tuning DistilBERT

In this exercise we will fine-tune the DistilBERT model to (hopefully) improve sentiment analysis performance.


---
### Exercise 2.1: Token Preprocessing

The first thing we need to do is *tokenize* our dataset splits -- we don't want to re-tokenize our inputs for every batch! Our current datasets return a dictionary with *strings*, but we want *input token ids* (i.e. the output of the tokenizer). This is easy enough to do by hand, but the Hugging Face `Dataset` class provides convenient, efficient, and *lazy* methods. See the documentation for [`Dataset.map`](https://huggingface.co/docs/datasets/v3.5.0/en/package_reference/main_classes#datasets.Dataset.map).

**Tip**: Verify that your new datasets are returning for every element: `text`, `label`, `intput_ids`, and `attention_mask`.

I use the map function to tokenize all the data in the train, validation and test set

In [ ]:
# Your code here.

def tokenize_function(dataset):
    return tokenizer(dataset['text'], truncation=True)

tokenized_train = ds['train'].map(tokenize_function, batched=True)
tokenized_train.set_format('pt', columns=['input_ids'], output_all_columns=True)


In [66]:
tokenized_train

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8530
})

In [67]:
tokenized_train['input_ids']

Column([tensor([  101,  1996,  2600,  2003, 16036,  2000,  2022,  1996,  7398,  2301,
         1005,  1055,  2047,  1000, 16608,  1000,  1998,  2008,  2002,  1005,
         1055,  2183,  2000,  2191,  1037, 17624,  2130,  3618,  2084,  7779,
        29058,  8625, 13327,  1010,  3744,  1011, 18856, 19513,  3158,  5477,
         4168,  2030,  7112, 16562,  2140,  1012,   102]), tensor([  101,  1996,  9882,  2135,  9603, 13633,  1997,  1000,  1996,  2935,
         1997,  1996,  7635,  1000, 11544,  2003,  2061,  4121,  2008,  1037,
         5930,  1997,  2616,  3685, 23613,  6235,  2522,  1011,  3213,  1013,
         2472,  2848,  4027,  1005,  1055,  4423,  4432,  1997,  1046,  1012,
         1054,  1012,  1054,  1012, 23602,  1005,  1055,  2690,  1011,  3011,
         1012,   102]), tensor([  101,  4621,  2021,  2205,  1011,  8915, 23267, 16012, 24330,   102]), tensor([  101,  2065,  2017,  2823,  2066,  2000,  2175,  2000,  1996,  5691,
         2000,  2031,  4569,  1010,  2001, 28518,

In [68]:
tokenized_train['attention_mask']

Column([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], ...])

In [ ]:
tokenized_val = ds['validation'].map(tokenize_function, batched=True)
tokenized_val.set_format('pt', columns=['input_ids'], output_all_columns=True)

In [71]:
tokenized_val

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1066
})

In [ ]:
tokenized_test = ds['test'].map(tokenize_function, batched=True)
tokenized_test.set_format('pt', columns=['input_ids'], output_all_columns=True)

In [118]:
tokenized_test

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1066
})


---
### Exercise 2.2: Setting up the Model to be Fine-tuned

In this exercise we need to prepare the base Distilbert model for fine-tuning for a *sequence classification task*. This means, at the very least, appending a new, randomly-initialized classification head connected to the `[CLS]` token of the last transformer layer. Luckily, HuggingFace already provides an `AutoModel` for just this type of instantiation: [`AutoModelForSequenceClassification`](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html#automodelforsequenceclassification). You will want you instantiate one of these for fine-tuning.

I instatiate the distilbert-base-uncased for classification

In [ ]:
from transformers import AutoModelForSequenceClassification

# Your code here.
cls_model = AutoModelForSequenceClassification.from_pretrained('distilbert/distilbert-base-uncased', num_labels=2) 

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [60]:
cls_model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [107]:
from transformers import DistilBertForSequenceClassification

cls_model = DistilBertForSequenceClassification.from_pretrained('distilbert/distilbert-base-uncased', num_labels=2) 

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [89]:
cls_model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(


---
### Exercise 2.3: Fine-tuning DistilBERT

Finally. In this exercise you should use a HuggingFace [`Trainer`](https://huggingface.co/docs/transformers/main/en/trainer) to fine-tune your model on the Rotten Tomatoes training split. Setting up the trainer will involve (at least):


1. Instantiating a [`DataCollatorWithPadding`](https://huggingface.co/docs/transformers/en/main_classes/data_collator) object which is what *actually* does your batch construction (by padding all sequences to the same length).
2. Writing an *evaluation function* that will measure the classification accuracy. This function takes a single argument which is a tuple containing `(logits, labels)` which you should use to compute classification accuracy (and maybe other metrics like F1 score, precision, recall) and return a `dict` with these metrics.  
3. Instantiating a [`TrainingArguments`](https://huggingface.co/docs/transformers/v4.51.1/en/main_classes/trainer#transformers.TrainingArguments) object using some reasonable defaults.
4. Instantiating a `Trainer` object using your train and validation splits, you data collator, and function to compute performance metrics.
5. Calling `trainer.train()`, waiting, waiting some more, and then calling `trainer.evaluate()` to see how it did.

**Tip**: When prototyping this laboratory I discovered the HuggingFace [Evaluate library](https://huggingface.co/docs/evaluate/en/index) which provides evaluation metrics. However I found it to have insufferable layers of abstraction and getting actual metrics computed. I suggest just using the Scikit-learn metrics...

I instatiate a dataCollatorWithPadding to construct the batches and pad every sequence to the same length

In [74]:
# Your code here.
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [76]:
data_collator

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

I set the parameters to finetne the model. The parameters that have worked the best for me are num_train_epochs= 15 e lr = 1e-6. If i use a bigger lr es 1e-5 it immediately overfits on the training set instead if i use a smaller lr es 1e-7 i need a lot more epochs to obtain the same results

In [106]:
training_args = TrainingArguments(
    output_dir='./output',
    learning_rate=1e-6,
    per_device_train_batch_size=48,
    per_device_eval_batch_size=48,
    num_train_epochs=15,
    use_cpu=False,
    save_strategy='epoch',
    report_to='wandb',
    logging_strategy="steps",
    logging_steps=1,
    do_eval=True,
    eval_strategy='epoch'
)

In [91]:
def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    labels = eval_pred.label_ids
    preds = logits.argmax(-1)

    report = classification_report(labels, preds, output_dict=True)

    return {
        "accuracy": report["accuracy"],
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"]
    }

In [108]:
trainer = Trainer(
   model=cls_model,
   args=training_args,
   train_dataset=tokenized_train,
   data_collator=data_collator,
   eval_dataset=tokenized_val,
   compute_metrics=compute_metrics,
)

In [109]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall
1,0.655629,0.656130,0.681051,0.677986,0.677986,0.688217,0.681051
2,0.557171,0.560583,0.771107,0.770926,0.770926,0.771969,0.771107
3,0.501220,0.470180,0.799250,0.799243,0.799243,0.799287,0.799250
4,0.423886,0.434792,0.806754,0.806580,0.806580,0.807864,0.806754
5,0.600420,0.416194,0.818011,0.817970,0.817970,0.818298,0.818011
6,0.613880,0.404927,0.817073,0.817046,0.817046,0.817262,0.817073
7,0.393074,0.398539,0.822702,0.822702,0.822702,0.822703,0.822702
8,0.282135,0.392961,0.830206,0.830206,0.830206,0.830208,0.830206
9,0.350268,0.390240,0.826454,0.826453,0.826453,0.826464,0.826454
10,0.469043,0.386980,0.831144,0.831142,0.831142,0.831163,0.831144


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2670, training_loss=0.4223970060491383, metrics={'train_runtime': 209.9321, 'train_samples_per_second': 609.483, 'train_steps_per_second': 12.718, 'total_flos': 1808549451508464.0, 'train_loss': 0.4223970060491383, 'epoch': 15.0})

In [119]:
metrics = trainer.evaluate(tokenized_test)

print(metrics)

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall
0.355050,0.398139,15,0.827392,0.827390,0.827390,0.827411,0.827392


{'eval_loss': 0.3981386125087738, 'eval_accuracy': 0.8273921200750469, 'eval_macro_f1': 0.8273896897055459, 'eval_weighted_f1': 0.8273896897055459, 'eval_macro_precision': 0.8274105599617, 'eval_macro_recall': 0.8273921200750469}


With finetuning I reach an accuracy of 83% on test set compared to the 80% of the baseline


---
---
## Exercise 3: Choose your Own Adventure

As promised, you should choose **one** of the following exercises to work. Well, at *least* one. If you want to do them all, that is also OK! Or if you want to propose something else as a third exercise, reach out to me on the Discord!


---
### Exercise 3.1: Efficient Fine-tuning for Sentiment Analysis

In Exercise 2 we fine-tuned the *entire* Distilbert model on Rotten Tomatoes. This is expensive, even for a small model. Find an *efficient* way to fine-tune Distilbert on the Rotten Tomatoes dataset (or some other dataset).

**Hint**: You could check out the [HuggingFace PEFT library](https://huggingface.co/docs/peft/en/index) for some state-of-the-art approaches that should "just work". How else might you go about making fine-tuning more efficient without having to change your training pipeline from above?

**Why choose this exercise?** PEFT techniques -- especially LoRA are the methods of choice for adapting models to new tasks.

In [10]:
# Your code here.


---
### Exercise 3.2: Fine-tuning a CLIP Model (harder)

Use a (small) CLIP model like [`openai/clip-vit-base-patch16`](https://huggingface.co/openai/clip-vit-base-patch16) and evaluate its zero-shot performance on a small image classification dataset like ImageNette or TinyImageNet. Fine-tune (using a parameter-efficient method!) the CLIP model to see how much improvement you can squeeze out of it.

**Note**: There are several ways to adapt the CLIP model; you could fine-tune the image encoder, the text encoder, or both. Or, you could experiment with prompt learning.

**Tip**: CLIP probably already works very well on ImageNet and ImageNet-like images. For extra fun, look for an image classification dataset with different image types (e.g. *sketches*).

**Why choose this exercise?** CLIP is probably the most widely used Vision-Language Model, and adapting it is a useful skill to master.

In [1]:
# Your code here.


---
### Exercise 3.3: A Text-to-image Retrieval System (hard, but not *too* hard)

Implement a simple text-to-image retrieval system with a simple user interface --- using, for example, [gradio](https://www.gradio.app/), or [Marimo](https://marimo.io/), or [Shiny](https://shiny.posit.co/). Your application should *index* (e.g. compute visual descriptors for) a small dataset of images like [Flickr8k](https://huggingface.co/datasets/jxie/flickr8k). It should provide a user interface with which a user can enter a short text prompt (e.g. "a photo of dogs playing in the snow") and then display the top-10 matching images from the indexed dataset.

Note that there is no following code block with "Your code here" for this exercise. You will definitely want to implement this outside of a Jupyter Notebook.

**Hint**: The **CLIP** model is practically *made* for just such an application.

**Why choose this exercise?** Well, this is a course on Deep Learning *Applications*, and this is your chance to *build* one!

---
---